# 1. 데이터 로드

In [ ]:
import sys
import os
import time

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

In [ ]:
train_file = 'train_set_tuples_with_note'
val_file = 'val_set_tuples_with_note'
test_file = 'test_set_tuples_with_note'

# This assumes MIMIC-III sepsis cohort has been extracted from https://github.com/microsoft/mimic_sepsis
full_data_file = os.path.join('./preprocessed_files/sepsis_final_data_withTimes.csv')

In [ ]:
full_zs = pd.read_csv(full_data_file)

## Determine the train, val, test split (70/15/15), stratified by patient outcome (died_within_48h_of_out_time)
temp = full_zs.groupby('traj')['r:reward'].sum()
y = temp.values
X = temp.index.values

In [ ]:
print(len(full_zs.columns))
full_zs.rename(columns={'m:text': 't:clinical_note'}, inplace=True)
full_zs.columns

In [ ]:
from collections import Counter
Counter(y)

# 2.개별 노트 데이터보고 요약 필요성 판단

In [ ]:
max_len = full_zs["t:clinical_note"].dropna().str.len().max()
print("Maximum length of t:clinical_note:", max_len)

In [ ]:
idx_max = full_zs["t:clinical_note"].str.len().idxmax()
# Extract the row at the selected index.
max_row = full_zs.loc[[idx_max]]
print(max_row['t:clinical_note'].values)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, stratify=y_test, test_size=0.5)

In [ ]:
# Drop unneeded meta features
full_zs = full_zs.drop(['m:timestep', 'm:icustayid'], axis=1)

In [ ]:
train_data = full_zs[full_zs['traj'].isin(X_train)]
trajectories = train_data['traj'].unique()

val_data = full_zs[full_zs['traj'].isin(X_val)]
val_trajectories = val_data['traj'].unique()

test_data = full_zs[full_zs['traj'].isin(X_test)]
test_trajectories = test_data['traj'].unique()

In [ ]:
dem_keep_cols = ['o:gender', 'o:mechvent', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:elixhauser', 'o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:Arterial_BE', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:cumulated_balance', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_step']
text_keep_cols = ['t:clinical_note']

In [ ]:
len(dem_keep_cols)+len(obs_keep_cols) + len(text_keep_cols)

In [ ]:
# Define the features of the full data
text_dim = 1
num_actions = 25
num_dem = 5 #demographics
num_obs = 42
num_notes = 1
horizon = 21
device = 'cpu'

In [ ]:
################################################################
#          FORMAT DATA FOR USE IN SEQUENTIAL MODELS
################################################################
text_keep_cols = ['t:clinical_note']
dem_keep_cols = ['o:gender', 'o:mechvent', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:elixhauser', 'o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:Arterial_BE', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:cumulated_balance', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_step']

note_cols = [i for i in train_data.columns if i in text_keep_cols]
dem_cols = [i for i in train_data.columns if i in dem_keep_cols]
obs_cols = [i for i in train_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in train_data.columns if i[:2] == 'a:']
rew_cols = [i for i in train_data.columns if i[:2] == 'r:']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(ac_cols) == 1, 'Multiple action columns are present when a single action column is expected'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

In [ ]:
## TRAINING DATA
#---------------------------------------------------------------------------
print("Converting Training Data")
print("--"*20)
# train_data[ac_col] = train_data[ac_col]
all_actions = train_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.int32)
except:
    raise ValueError('Actions are expected to be integers, but are not.')

In [ ]:
all_actions

In [ ]:
# if not all(all_actions == np.arange(num_actions, dtype=np.int32)):
    # print(Font.red + 'Some actions are missing from data or all action space not properly defined.' + Font.end)
data_trajectory = {}
data_trajectory['text_cols'] = note_cols
data_trajectory['dem_cols'] = dem_cols
data_trajectory['obs_cols'] = obs_cols
data_trajectory['ac_col']  = ac_col
data_trajectory['rew_col'] = rew_col
data_trajectory['num_actions'] = num_actions
data_trajectory['obs_dim'] = len(obs_cols)
data_trajectory['traj'] = {}
data_trajectory['pos_traj'] = []
data_trajectory['neg_traj'] = []

for i in trajectories:
    # bar.update()
    traj_i = train_data[train_data['traj'] == i].sort_values(by='step')
    data_trajectory['traj'][i] = {}
    data_trajectory['traj'][i]['note'] = traj_i[note_cols].values
    data_trajectory['traj'][i]['dem'] = torch.Tensor(traj_i[dem_cols].values).to('cpu')
    data_trajectory['traj'][i]['obs'] = torch.Tensor(traj_i[obs_cols].values).to('cpu')
    data_trajectory['traj'][i]['actions'] = torch.Tensor(traj_i[ac_col].values.astype(np.int32)).to('cpu').long()
    data_trajectory['traj'][i]['rewards'] = torch.Tensor(traj_i[rew_col].values).to('cpu')
    if sum(traj_i[rew_col].values) > 0:
        data_trajectory['pos_traj'].append(i)
    else:
        data_trajectory['neg_traj'].append(i)

In [ ]:
notes = np.full((len(trajectories), horizon, num_notes), "None", dtype=object)
observations = torch.zeros((len(trajectories), horizon, num_obs))
demographics = torch.zeros((len(trajectories), horizon, num_dem)) 
actions = torch.zeros((len(trajectories), horizon-1, num_actions))
lengths = torch.zeros((len(trajectories)), dtype=torch.int)
times = torch.zeros((len(trajectories), horizon))
rewards = torch.zeros((len(trajectories), horizon))
action_temp = torch.eye(25) #one-hot encoded action (25,25)

In [ ]:
for ii, traj in enumerate(trajectories):
    text = data_trajectory['traj'][traj]['note']
    obs = data_trajectory['traj'][traj]['obs']
    dem = data_trajectory['traj'][traj]['dem']
    action = data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    
    lengths[ii] = length
    temp = action_temp[action].squeeze(1)

    notes[ii] = np.concatenate((text, np.full((horizon-length, 1), 'None')), axis=0)
    observations[ii] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    demographics[ii] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    actions[ii] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    times[ii] = torch.Tensor(range(horizon))
    rewards[ii] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))

In [ ]:
notes = notes[lengths>1.0]
actions = actions[lengths>1.0].to(device)
observations = observations[lengths>1.0].to(device)
demographics = demographics[lengths>1.0].to(device)
times = times[lengths>1.0].to(device)
rewards = rewards[lengths>1.0].to(device)
lengths = lengths[lengths>1.0].to(device)

In [ ]:
print(notes.shape, demographics.shape, observations.shape, actions.shape, lengths.shape, times.shape, rewards.shape)

In [ ]:
notes

In [ ]:
for idx, value in np.ndenumerate(notes):
    # Handle numeric values.
    if isinstance(value, (int, float, np.integer, np.floating)):
        # Replace NaN or zero values with 'None'.
        if (isinstance(value, (float, np.floating)) and np.isnan(value)) or (value == 0):
            notes[idx] = 'None'
    # For string values, handle the literal string '0'.
    elif isinstance(value, str) and value.strip() == '0':
        notes[idx] = 'None'

In [ ]:
notes

In [ ]:
print("Saving off tuples")
print("..."*20)
torch.save((notes, demographics,observations,actions,lengths,times,rewards),os.path.join("./tuples",train_file))

In [ ]:
print(notes[0])

In [ ]:
print(demographics[0])
print(": 7개 길이의 데이터만 보유")

In [ ]:
print(lengths[0])
print(": 0번째 traj이 가지고 있는 실제 traj길이 저장")

In [ ]:
print(len(data_trajectory['traj']))
print(len(data_trajectory['pos_traj']))
print(len(data_trajectory['neg_traj']))

In [ ]:
## Validation DATA
#---------------------------------------------------------------------------
print("Converting Validation Data")
print("="*20)
val_data_trajectory = {}
val_data_trajectory['text_cols'] = note_cols
val_data_trajectory['obs_cols'] = obs_cols
val_data_trajectory['dem_cols'] = dem_cols
val_data_trajectory['ac_col']  = ac_col
val_data_trajectory['rew_col'] = rew_col
val_data_trajectory['num_actions'] = num_actions
val_data_trajectory['obs_dim'] = len(obs_cols)
val_data_trajectory['traj'] = {}
val_data_trajectory['pos_traj'] = []
val_data_trajectory['neg_traj'] = []


In [ ]:
for j in val_trajectories:
    traj_j = val_data[val_data['traj']==j].sort_values(by='step')
    val_data_trajectory['traj'][j] = {}
    val_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    val_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    val_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    val_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    val_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        val_data_trajectory['pos_traj'].append(j)
    else:
        val_data_trajectory['neg_traj'].append(j)

In [ ]:
val_notes = np.full((len(val_trajectories), horizon, num_notes), "None", dtype=object)
val_obs = torch.zeros((len(val_trajectories), horizon, num_obs))
val_dem = torch.zeros((len(val_trajectories), horizon, num_dem))
val_actions = torch.zeros((len(val_trajectories), horizon-1, num_actions))
val_lengths = torch.zeros((len(val_trajectories)), dtype=torch.int)
val_times = torch.zeros((len(val_trajectories),horizon))
val_rewards = torch.zeros((len(val_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(val_trajectories):
    text = val_data_trajectory['traj'][traj]['note']
    obs = val_data_trajectory['traj'][traj]['obs']
    dem = val_data_trajectory['traj'][traj]['dem']
    action = val_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = val_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    val_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    val_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), 'None')), axis=0)
    val_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    val_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    val_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    val_times[jj] = torch.Tensor(range(horizon))
    val_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))


# Eliminate single transition trajectories...
val_notes = val_notes[val_lengths>1.0]
val_actions = val_actions[val_lengths>1.0].to(device)
val_obs = val_obs[val_lengths>1.0].to(device)
val_dem = val_dem[val_lengths>1.0].to(device)
val_times = val_times[val_lengths>1.0].to(device)
val_rewards = val_rewards[val_lengths>1.0].to(device)
val_lengths = val_lengths[val_lengths>1.0].to(device)

In [ ]:
print(len(val_data_trajectory['traj']))
print(len(val_data_trajectory['pos_traj']))
print(len(val_data_trajectory['neg_traj']))

In [ ]:
print(val_notes.shape, val_actions.shape)

In [ ]:
print(val_notes.shape, val_dem.shape, val_obs.shape, val_actions.shape, val_lengths.shape, val_times.shape, val_rewards.shape)

In [ ]:
val_notes

In [ ]:
for idx, value in np.ndenumerate(val_notes):
    # Handle numeric values.
    if isinstance(value, (int, float, np.integer, np.floating)):
        # Replace NaN or zero values with 'None'.
        if (isinstance(value, (float, np.floating)) and np.isnan(value)) or (value == 0):
            val_notes[idx] = 'None'
    # For string values, handle the literal string '0'.
    elif isinstance(value, str) and value.strip() == '0':
        val_notes[idx] = 'None'

In [ ]:
val_notes

In [ ]:
print("Saving off tuples")
print("..."*20)
torch.save((val_notes, val_dem,val_obs,val_actions,val_lengths,val_times,val_rewards),os.path.join("./tuples",val_file))

In [ ]:
## Test DATA
#---------------------------------------------------------------------------
print("Converting Test Data")
print("+"*20)
test_data_trajectory = {}
test_data_trajectory['text_cols'] = note_cols
test_data_trajectory['obs_cols'] = obs_cols
test_data_trajectory['dem_cols'] = dem_cols
test_data_trajectory['ac_col']  = ac_col
test_data_trajectory['rew_col'] = rew_col
test_data_trajectory['num_actions'] = num_actions
test_data_trajectory['obs_dim'] = len(obs_cols)
test_data_trajectory['traj'] = {}
test_data_trajectory['pos_traj'] = []
test_data_trajectory['neg_traj'] = []

for j in test_trajectories:
    traj_j = test_data[test_data['traj']==j].sort_values(by='step')
    test_data_trajectory['traj'][j] = {}
    test_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    test_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    test_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        test_data_trajectory['pos_traj'].append(j)
    else:
        test_data_trajectory['neg_traj'].append(j)

test_notes = np.full((len(test_trajectories), horizon, num_notes), "0", dtype=object)
test_obs = torch.zeros((len(test_trajectories), horizon, num_obs))
test_dem = torch.zeros((len(test_trajectories), horizon, num_dem))
test_actions = torch.zeros((len(test_trajectories), horizon-1, num_actions))
test_lengths = torch.zeros((len(test_trajectories)), dtype=torch.int)
test_times = torch.zeros((len(test_trajectories), horizon))
test_rewards = torch.zeros((len(test_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(test_trajectories):
    text = test_data_trajectory['traj'][traj]['note']
    obs = test_data_trajectory['traj'][traj]['obs']
    dem = test_data_trajectory['traj'][traj]['dem']
    action = test_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = test_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    test_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    test_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), '0')), axis=0)
    test_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    test_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    test_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    test_times[jj] = torch.Tensor(range(horizon))
    test_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))



In [ ]:
# Eliminate single transition trajectories...
test_notes = test_notes[test_lengths>1.0]
test_actions = test_actions[test_lengths>1.0].to(device)
test_obs = test_obs[test_lengths>1.0].to(device)
test_dem = test_dem[test_lengths>1.0].to(device)
test_times = test_times[test_lengths>1.0].to(device)
#test_mortality = test_mortality[test_lengths>1.0].to(device)
test_rewards = test_rewards[test_lengths>1.0].to(device)
test_lengths = test_lengths[test_lengths>1.0].to(device)

In [ ]:
print(test_notes.shape, test_actions.shape, test_obs.shape, test_dem.shape, test_times.shape, test_rewards.shape, test_lengths.shape)

In [ ]:
for idx, value in np.ndenumerate(test_notes):
    # Handle numeric values.
    if isinstance(value, (int, float, np.integer, np.floating)):
        # Replace NaN or zero values with 'None'.
        if (isinstance(value, (float, np.floating)) and np.isnan(value)) or (value == 0):
            test_notes[idx] = 'None'
    # For string values, handle the literal string '0'.
    elif isinstance(value, str) and value.strip() == '0':
        test_notes[idx] = 'None'

In [ ]:
test_notes

In [ ]:
print(test_notes.shape, test_actions.shape, test_obs.shape, test_dem.shape, test_times.shape, test_rewards.shape, test_lengths.shape)

In [ ]:
torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./tuples",test_file))